In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score, precision_recall_curve

1. Загрузите файл classification.csv. В нем записаны истинные классы объектов выборки (колонка true) и ответы некоторого классификатора (колонка predicted).

In [ ]:
df = pd.read_csv('classification.csv')
y_true = df['true']
y_pred = df['pred']
df.head()

2. Заполните таблицу ошибок классификации:

|  | Actual Positive | Actual Negative |
|---|---|---|
| Predicted Positive | TP | FP |
| Predicted Negative | FN | TN |

Для этого подсчитайте величины TP, FP, FN и TN согласно их определениям. Например, FP — это количество объектов, имеющих класс 0, но отнесенных алгоритмом к классу 1. Ответ в данном вопросе — четыре числа через пробел.

In [ ]:
TP = int(((y_true == 1) & (y_pred == 1)).sum())
FP = int(((y_true == 0) & (y_pred == 1)).sum())
FN = int(((y_true == 1) & (y_pred == 0)).sum())
TN = int(((y_true == 0) & (y_pred == 0)).sum())
print(f'TP={TP}  FP={FP}  FN={FN}  TN={TN}')
print(f'Ответ: {TP} {FP} {FN} {TN}')

3. Посчитайте основные метрики качества классификатора:
- Accuracy (доля верно угаданных) — sklearn.metrics.accuracy_score
- Precision (точность) — sklearn.metrics.precision_score
- Recall (полнота) — sklearn.metrics.recall_score
- F-мера — sklearn.metrics.f1_score

In [ ]:
acc  = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec  = recall_score(y_true, y_pred)
f1   = f1_score(y_true, y_pred)
print(f'Accuracy : {acc:.2f}')
print(f'Precision: {prec:.2f}')
print(f'Recall   : {rec:.2f}')
print(f'F1       : {f1:.2f}')

4. Загрузите файл scores.csv. В нем записаны истинные классы и значения степени принадлежности положительному классу для каждого из четырёх классификаторов:
- score_logreg — вероятность положительного класса (логистическая регрессия)
- score_svm — отступ от разделяющей поверхности (SVM)
- score_knn — взвешенная сумма классов соседей (метрический алгоритм)
- score_tree — доля положительных объектов в листе (решающее дерево)

In [ ]:
scores = pd.read_csv('scores.csv')
scores.head()

5. Посчитайте площадь под ROC-кривой для каждого классификатора. Какой классификатор имеет наибольшее значение метрики AUC-ROC?

In [ ]:
y2 = scores['true']
score_cols = ['score_logreg', 'score_svm', 'score_knn', 'score_tree']

auc_scores = {}
for col in score_cols:
    auc = roc_auc_score(y2, scores[col])
    auc_scores[col] = auc
    print(f'{col}: AUC-ROC = {auc:.4f}')

best_auc_col = max(auc_scores, key=auc_scores.get)
print(f'\nЛучший AUC-ROC: {best_auc_col} ({auc_scores[best_auc_col]:.4f})')

6. Какой классификатор достигает наибольшей точности (Precision) при полноте (Recall) не менее 70%?

Используем sklearn.metrics.precision_recall_curve для получения всех точек кривой, затем находим максимальную точность среди точек с recall >= 0.7.

In [ ]:
best_col = None
best_prec_val = 0

for col in score_cols:
    precision, recall, _ = precision_recall_curve(y2, scores[col])
    mask = recall >= 0.7
    if mask.any():
        max_prec = precision[mask].max()
        print(f'{col}: max precision при recall>=0.7 = {max_prec:.2f}')
        if max_prec > best_prec_val:
            best_prec_val = max_prec
            best_col = col

print(f'\nЛучший классификатор: {best_col}')
print(f'Значение точности: {round(best_prec_val, 2)}')